# Dynamic Waste Collection Route Optimization
## Complete Demo Notebook

This notebook demonstrates the complete workflow for predicting bin fill levels and optimizing collection routes.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.preprocessing import BinDataPreprocessor
from src.data.feature_engineering import BinFeatureEngineer
from src.models.random_forest_model import BinFillPredictor
from src.optimization.route_optimizer import WasteCollectionRouter
from src.visualization.map_viz import RouteMapVisualizer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")

## 2. Data Loading and Exploration

In [ ]:
# Load data
preprocessor = BinDataPreprocessor()
data = preprocessor.load_data('../data/raw/smart-bins-argyle-square.csv')

# Display basic info
print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {data.columns.tolist()}")
data.head()

In [ ]:
# Data statistics
print("Fill Percentage Statistics:")
print(data['fill_percentage'].describe())

print(f"\nNumber of unique bins: {data['bin_id'].nunique()}")
print(f"Date range: {data['timestamp'].min()} to {data['timestamp'].max()}")

In [ ]:
# Visualize fill level distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(data['fill_percentage'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Fill Percentage (%)')
plt.ylabel('Frequency')
plt.title('Distribution of Fill Levels')

plt.subplot(1, 2, 2)
data.groupby('bin_id')['fill_percentage'].mean().plot(kind='bar')
plt.xlabel('Bin ID')
plt.ylabel('Average Fill (%)')
plt.title('Average Fill Level by Bin')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Clean data
clean_data = preprocessor.clean_data()

print(f"✓ Data cleaned: {len(clean_data)} records")
clean_data.info()

## 4. Feature Engineering

In [ ]:
# Create features
engineer = BinFeatureEngineer()
featured_data = engineer.create_all_features(clean_data)

print(f"\nTotal features: {len(featured_data.columns)}")
print(f"\nFeature columns:")
print(featured_data.columns.tolist())

In [ ]:
# Visualize some engineered features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Hourly pattern
featured_data.groupby('hour')['fill_percentage'].mean().plot(ax=axes[0, 0], marker='o')
axes[0, 0].set_title('Average Fill Level by Hour')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Fill %')

# Daily pattern
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_avg = featured_data.groupby('day_of_week')['fill_percentage'].mean()
axes[0, 1].bar(range(7), daily_avg.values)
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(day_names)
axes[0, 1].set_title('Average Fill Level by Day of Week')
axes[0, 1].set_ylabel('Fill %')

# Fill rate
axes[1, 0].hist(featured_data['fill_rate'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of Fill Rates')
axes[1, 0].set_xlabel('Fill Rate (% per hour)')
axes[1, 0].set_ylabel('Frequency')

# Rolling mean
sample_bin = featured_data[featured_data['bin_id'] == featured_data['bin_id'].iloc[0]].head(100)
axes[1, 1].plot(sample_bin['fill_percentage'], label='Actual', alpha=0.7)
axes[1, 1].plot(sample_bin['fill_percentage_rolling_mean_6'], label='Rolling Mean (6h)', linewidth=2)
axes[1, 1].set_title('Fill Level with Rolling Average')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Fill %')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 5. Model Training - Random Forest

In [ ]:
# Prepare ML data
X_train, y_train, X_test, y_test = engineer.prepare_ml_data(featured_data, test_size=0.2)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Features: {X_train.shape[1]}")

In [ ]:
# Train Random Forest model
predictor = BinFillPredictor(n_estimators=100, max_depth=20)
predictor.train(X_train, y_train)

In [ ]:
# Evaluate model
metrics = predictor.evaluate(X_test, y_test)

print("\n📊 Model Performance:")
for metric, value in metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

In [ ]:
# Feature importance
top_features = predictor.feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Prediction visualization
predictions = predictor.predict(X_test)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, predictions, alpha=0.5, s=10)
plt.plot([0, 100], [0, 100], 'r--', lw=2)
plt.xlabel('Actual Fill %')
plt.ylabel('Predicted Fill %')
plt.title('Actual vs Predicted')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y_test - predictions
plt.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title('Residuals Distribution')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

## 6. Predictions for Current Bins

In [ ]:
# Get latest bin status
latest_status = preprocessor.get_latest_status()

print(f"Latest status for {len(latest_status)} bins")
print(f"\nCurrent fill levels:")
print(latest_status[['bin_id', 'fill_percentage', 'latitude', 'longitude']].head(10))

In [ ]:
# Make predictions
latest_featured = engineer.create_all_features(
    preprocessor.data[preprocessor.data['bin_id'].isin(latest_status['bin_id'])]
).groupby('bin_id').tail(1)

exclude_cols = ['bin_id', 'timestamp', 'fill_percentage', 'bin_type', 'latitude', 'longitude']
feature_cols = [col for col in latest_featured.columns if col not in exclude_cols]

X_predict = latest_featured[feature_cols]
X_predict_scaled = engineer.scaler.transform(X_predict)

predictions = predictor.predict(X_predict_scaled)
latest_status['predicted_fill'] = predictions

print(f"\n✓ Predictions completed")
print(f"Average predicted fill: {predictions.mean():.2f}%")
print(f"\nTop 10 bins by predicted fill:")
print(latest_status.nlargest(10, 'predicted_fill')[['bin_id', 'fill_percentage', 'predicted_fill']])

## 7. Route Optimization

In [ ]:
# Initialize router
router = WasteCollectionRouter(
    depot_location=(51.5294, -0.1194),
    vehicle_capacity=10,
    num_vehicles=3
)

# Get bins needing collection
bins_to_collect = router.get_bins_needing_collection(latest_status, threshold=80)

print(f"\nBins needing collection (≥80%):")
print(bins_to_collect[['bin_id', 'predicted_fill', 'latitude', 'longitude']])

In [ ]:
# Optimize routes
if len(bins_to_collect) > 0:
    result = router.optimize_routes(bins_to_collect, time_limit_seconds=30)
    
    print("\n" + "="*50)
    print("OPTIMIZATION RESULTS")
    print("="*50)
    print(f"Total distance: {result['total_distance_km']} km")
    print(f"Vehicles used: {result['num_vehicles_used']}")
    print(f"Bins collected: {result['total_bins_collected']}")
    print(f"\nRoute details:")
    for route in result['routes']:
        print(f"  Vehicle {route['vehicle_id']}: {route['num_bins']} bins, {route['distance_km']} km")
else:
    print("\nNo bins need collection at this time.")
    result = None

## 8. Visualization

In [ ]:
# Create map visualizer
visualizer = RouteMapVisualizer(center_location=(51.5294, -0.1194))

# Visualize bin status
visualizer.visualize_bin_status(latest_status, save_path='../output/maps/bin_status.html')

print("✓ Bin status map saved to output/maps/bin_status.html")

In [ ]:
# Visualize optimized routes
if result:
    visualizer.visualize_routes(
        result,
        bins_to_collect,
        (51.5294, -0.1194),
        save_path='../output/maps/optimized_routes.html'
    )
    print("✓ Route map saved to output/maps/optimized_routes.html")
    print("\nOpen the HTML files in your browser to view the interactive maps!")

## 9. Summary Statistics

In [ ]:
# Create summary dataframe
summary = pd.DataFrame({
    'Metric': [
        'Total Bins',
        'Avg Current Fill',
        'Avg Predicted Fill',
        'Bins ≥80%',
        'Model RMSE',
        'Model R²',
        'Vehicles Used',
        'Total Distance',
        'Avg Distance/Bin'
    ],
    'Value': [
        len(latest_status),
        f"{latest_status['fill_percentage'].mean():.1f}%",
        f"{latest_status['predicted_fill'].mean():.1f}%",
        len(bins_to_collect) if result else 0,
        f"{metrics['rmse']:.2f}%",
        f"{metrics['r2']:.4f}",
        result['num_vehicles_used'] if result else 0,
        f"{result['total_distance_km']:.2f} km" if result else "N/A",
        f"{result['total_distance_km']/result['total_bins_collected']:.2f} km" if result and result['total_bins_collected'] > 0 else "N/A"
    ]
})

print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
print(summary.to_string(index=False))

## 10. Save Model

In [ ]:
# Save trained model
predictor.save_model('../models/random_forest_model.pkl')
print("✓ Model saved successfully")

## Conclusion

This notebook demonstrated:
1. ✅ Loading and preprocessing sensor data
2. ✅ Engineering time-series features
3. ✅ Training a Random Forest prediction model
4. ✅ Making predictions for current bin status
5. ✅ Optimizing collection routes using OR-Tools
6. ✅ Visualizing results on interactive maps

Next steps:
- Run the Streamlit dashboard: `streamlit run app.py`
- Experiment with LSTM model for time series
- Try different route optimization parameters
- Integrate real-time data feeds